# Running LiteSpecFormer on gift-eval benchmark

**LiteSpecFormer** is a lightweight channel-independent Transformer for zero-shot forecasting. Originally built for **wireless spectrum prediction** ([LSPD](https://huggingface.co/datasets/FlowVortex/Large-Spectrum-Prediction-Dataset), 18B timestamps), it is evaluated here on the general-purpose [GIFT-Eval](https://huggingface.co/spaces/Salesforce/GIFT-Eval) benchmark through a GluonTS-compatible predictor wrapper.

| | |
|---|---|
| **Model** | [`FlowVortex/LiteSpecFormer-1.0-36M`](https://huggingface.co/FlowVortex/LiteSpecFormer-1.0-36M) |
| **Project** | [FlowVortex/LiteSpecFormer](https://github.com/FlowVortex/LiteSpecFormer) |

This notebook reproduces GIFT-Eval leaderboard results. For everyday inference, see the [Quickstart demo](https://github.com/FlowVortex/LiteSpecFormer/blob/main/notebooks/demo.ipynb) in the LiteSpecFormer repo.

## Setup

**Install**

```bash
pip install litespecformer
```

Please install the dependencies according to the requirements in the GIFT-EVAL project README first, and then install litespecformer via pip.



Make sure you download the gift-eval benchmark and set the `GIFT-EVAL` environment variable correctly before running this notebook:

```bash
export GIFT_EVAL=/path/to/gift-eval-dataset
```

**Dataset selection** — edit `SHORT_DATASETS` and `MED_LONG_DATASETS` in the configuration cell. Defaults use two datasets as a quick smoke test.

In [13]:
import json
import logging
import os
import warnings

import pandas as pd
import torch
from datasets import Dataset as HFDataset
from dotenv import load_dotenv
from gluonts.ev.metrics import (
    MAE,
    MAPE,
    MASE,
    MSE,
    MSIS,
    ND,
    NRMSE,
    RMSE,
    SMAPE,
    MeanWeightedSumQuantileLoss,
)
from gluonts.itertools import batcher
from gluonts.model import Forecast, evaluate_forecasts
from gluonts.model.forecast import QuantileForecast
from gluonts.time_feature import get_seasonality
from litespecformer import LiteSpecFormerPipeline
from tqdm import tqdm

from gift_eval.data import Dataset


def normalize_freq(freq) -> str:
    """Convert numpy freq strings to pandas >= 2.2 compatible aliases."""
    freq = str(freq)
    unit_map = {"H": "h", "T": "min", "S": "s", "U": "us", "N": "ns"}
    if freq in unit_map:
        return unit_map[freq]
    for old, new in (("T", "min"), ("H", "h"), ("S", "s")):
        if len(freq) > len(old) and freq.endswith(old) and freq[: -len(old)].isdigit():
            return freq[: -len(old)] + new
    return freq


def _normalized_freq(self) -> str:
    return normalize_freq(self.hf_dataset[0]["freq"])


Dataset.freq = property(_normalized_freq)

os.environ["TQDM_DISABLE"] = "1"
os.environ.setdefault("HF_ENDPOINT", "https://hf-mirror.com")

load_dotenv()

warnings.filterwarnings(
    "ignore",
    category=FutureWarning,
    message=r".*is deprecated and will be removed.*",
)
warnings.filterwarnings(
    "ignore",
    category=UserWarning,
    message=r".*We recommend keeping prediction length.*",
)


class _WarningFilter(logging.Filter):
    def __init__(self, text_to_filter: str):
        super().__init__()
        self.text_to_filter = text_to_filter

    def filter(self, record: logging.LogRecord) -> bool:
        return self.text_to_filter not in record.getMessage()


logger = logging.getLogger("LiteSpecFormer for Gift-Eval")
logger.setLevel(logging.INFO)

logging.getLogger("gluonts.model.forecast").addFilter(
    _WarningFilter("The mean prediction is not stored in the forecast data")
)

In [7]:
# --- Model & output ---
MODEL_NAME = "FlowVortex/LiteSpecFormer-1.0-36M"
OUTPUT_DIR = "../results/litespecformer/all_results.csv"
DEVICE_MAP = "cuda"
BATCH_SIZE = 512
MIN_PAST = 32  # minimum context length; shorter series are left-padded with NaN

PRETTY_MODEL_NAME = {
    MODEL_NAME: "LiteSpecFormer",
}

PRETTY_DATASET_NAMES = {
    "saugeenday": "saugeen",
    "temperature_rain_with_missing": "temperature_rain",
    "kdd_cup_2018_with_missing": "kdd_cup_2018",
    "car_parts_with_missing": "car_parts",
}

# --- Datasets ---
# SHORT_DATASETS = "m4_yearly m4_quarterly m4_monthly m4_weekly m4_daily m4_hourly electricity/15T electricity/H electricity/D electricity/W solar/10T solar/H solar/D solar/W hospital covid_deaths us_births/D us_births/M us_births/W saugeenday/D saugeenday/M saugeenday/W temperature_rain_with_missing kdd_cup_2018_with_missing/H kdd_cup_2018_with_missing/D car_parts_with_missing restaurant hierarchical_sales/D hierarchical_sales/W LOOP_SEATTLE/5T LOOP_SEATTLE/H LOOP_SEATTLE/D SZ_TAXI/15T SZ_TAXI/H M_DENSE/H M_DENSE/D ett1/15T ett1/H ett1/D ett1/W ett2/15T ett2/H ett2/D ett2/W jena_weather/10T jena_weather/H jena_weather/D bitbrains_fast_storage/5T bitbrains_fast_storage/H bitbrains_rnd/5T bitbrains_rnd/H bizitobs_application bizitobs_service bizitobs_l2c/5T bizitobs_l2c/H"
SHORT_DATASETS = "m4_weekly"

# MED_LONG_DATASETS = "electricity/15T electricity/H solar/10T solar/H kdd_cup_2018_with_missing/H LOOP_SEATTLE/5T LOOP_SEATTLE/H SZ_TAXI/15T M_DENSE/H ett1/15T ett1/H ett2/15T ett2/H jena_weather/10T jena_weather/H bitbrains_fast_storage/5T bitbrains_rnd/5T bizitobs_application bizitobs_service bizitobs_l2c/5T bizitobs_l2c/H"
MED_LONG_DATASETS = "bizitobs_l2c/H"

ALL_DATASETS = sorted(set(SHORT_DATASETS.split() + MED_LONG_DATASETS.split()))
MED_LONG_DATASET_SET = set(MED_LONG_DATASETS.split())

with open("dataset_properties.json") as f:
    DATASET_PROPERTIES = json.load(f)

QUANTILE_LEVELS = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]

METRICS = [
    MSE(forecast_type="mean"),
    MSE(forecast_type=0.5),
    MAE(),
    MASE(),
    MAPE(),
    SMAPE(),
    MSIS(),
    RMSE(),
    NRMSE(),
    ND(),
    MeanWeightedSumQuantileLoss(quantile_levels=QUANTILE_LEVELS),
]

## Predictor

`LiteSpecFormerPredictor` wraps `LiteSpecFormerPipeline` as a GluonTS `Forecast` producer. Context shorter than `MIN_PAST` is left-padded with NaN before inference.

In [14]:
class LiteSpecFormerPredictor:
    """GluonTS-compatible wrapper around LiteSpecFormerPipeline."""

    def __init__(
        self,
        model_name: str,
        prediction_length: int,
        batch_size: int = 512,
        quantile_levels: list[float] | None = None,
        min_past: int = MIN_PAST,
        device_map: str = DEVICE_MAP,
    ):
        self.pipeline = LiteSpecFormerPipeline.from_pretrained(
            model_name,
            device_map=device_map,
        )
        assert isinstance(self.pipeline, LiteSpecFormerPipeline)

        self.prediction_length = prediction_length
        self.batch_size = batch_size
        self.quantile_levels = quantile_levels or QUANTILE_LEVELS
        self.min_past = min_past

    @staticmethod
    def _left_pad_to_min_length(target: torch.Tensor, min_past: int) -> torch.Tensor:
        """Left-pad with NaN when the time dimension is shorter than min_past."""
        seq_len = target.shape[-1]
        if seq_len >= min_past:
            return target

        pad_len = min_past - seq_len
        padding = torch.full(
            (*target.shape[:-1], pad_len),
            fill_value=torch.nan,
            dtype=target.dtype,
        )
        return torch.cat([padding, target], dim=-1)

    def predict(self, test_data_input) -> list[Forecast]:
        forecasts: list[Forecast] = []

        for batch in tqdm(
            batcher(test_data_input, batch_size=self.batch_size),
            disable=True,
        ):
            context = [
                self._left_pad_to_min_length(
                    torch.tensor(entry["target"]), self.min_past
                )
                for entry in batch
            ]
            hf_dataset = HFDataset.from_list([{"target": item} for item in context])
            hf_dataset.set_format("torch", device=DEVICE_MAP)

            quantiles, _ = self.pipeline.predict_quantiles(
                inputs=hf_dataset,
                prediction_length=self.prediction_length,
                batch_size=self.batch_size,
            )
            # [batch, quantiles, prediction_length]
            outputs = (
                torch.stack(quantiles).detach().cpu().float().numpy().transpose(0, 2, 1)
            )

            for forecast_array, ts in zip(outputs, batch):
                forecasts.append(
                    QuantileForecast(
                        forecast_arrays=forecast_array,
                        forecast_keys=list(map(str, self.quantile_levels)),
                        start_date=ts["start"] + len(ts["target"]),
                    )
                )

        return forecasts

## Evaluation

Run LiteSpecFormer on each `(dataset, term)` pair and save metrics to `../results/litespecformer/all_results.csv`.

- **Row format:** `{dataset}/{freq}/{term}` — see [README](../README.md)
- **Terms:** `short` for all datasets; `medium` / `long` only for entries listed in `MED_LONG_DATASETS`

In [15]:
ALL_DATASETS

['bizitobs_l2c/H', 'm4_weekly']

In [16]:
def resolve_dataset_config(ds_name: str, term: str) -> tuple[str, str, str]:
    """Return (ds_key, ds_freq, ds_config) for a raw dataset name and term."""
    if "/" in ds_name:
        raw_key, ds_freq = ds_name.split("/", maxsplit=1)
        ds_key = PRETTY_DATASET_NAMES.get(raw_key.lower(), raw_key.lower())
    else:
        ds_key = PRETTY_DATASET_NAMES.get(ds_name.lower(), ds_name.lower())
        ds_freq = DATASET_PROPERTIES[ds_key]["frequency"]

    return ds_key, ds_freq, f"{ds_key}/{ds_freq}/{term}"


def evaluate_on_dataset(
    ds_name: str,
    ds_term: str,
    batch_size: int = BATCH_SIZE,
) -> list[dict]:
    to_univariate = (
        Dataset(name=ds_name, term=ds_term, to_univariate=False).target_dim != 1
    )
    dataset = Dataset(name=ds_name, term=ds_term, to_univariate=to_univariate)
    season_length = get_seasonality(dataset.freq)

    logger.info(f"Dataset size: {len(dataset.test_data)}")

    predictor = LiteSpecFormerPredictor(
        model_name=MODEL_NAME,
        prediction_length=dataset.prediction_length,
        batch_size=batch_size,
    )
    forecasts = predictor.predict(dataset.test_data.input)

    return (
        evaluate_forecasts(
            forecasts,
            test_data=dataset.test_data,
            metrics=METRICS,
            batch_size=1024,
            axis=None,
            mask_invalid_label=True,
            allow_nan_forecast=False,
            seasonality=season_length,
        )
        .reset_index(drop=True)
        .to_dict(orient="records")
    )


result_rows = []
for ds_num, ds_name in enumerate(ALL_DATASETS):
    logger.info(f"Processing dataset: {ds_name} ({ds_num + 1} of {len(ALL_DATASETS)})")

    for term in ("short", "medium", "long"):
        if term in ("medium", "long") and ds_name not in MED_LONG_DATASET_SET:
            continue

        ds_key, ds_freq, ds_config = resolve_dataset_config(ds_name, term)
        props = DATASET_PROPERTIES[ds_key]

        logger.info(f"Generating forecasts for {ds_config}")
        metrics = evaluate_on_dataset(ds_name=ds_name, ds_term=term)

        result_rows.append(
            {
                "dataset": ds_config,
                "model": PRETTY_MODEL_NAME.get(MODEL_NAME, MODEL_NAME),
                **{f"eval_metrics/{k}": v for k, v in metrics[0].items()},
                "domain": props["domain"],
                "num_variates": props["num_variates"],
            }
        )

42it [00:00, 564.12it/s]
7it [00:00, 333.32it/s]
7it [00:00, 346.04it/s]
359it [00:00, 571.60it/s]


In [17]:
results_df = pd.DataFrame(result_rows).sort_values(by="dataset")
results_df.to_csv(OUTPUT_DIR, index=False)
logger.info(f"Results written to {OUTPUT_DIR}")
results_df

,dataset,model,eval_metrics/MSE[mean],eval_metrics/MSE[0.5],eval_metrics/MAE[0.5],eval_metrics/MASE[0.5],eval_metrics/MAPE[0.5],eval_metrics/sMAPE[0.5],eval_metrics/MSIS,eval_metrics/RMSE[mean],eval_metrics/NRMSE[mean],eval_metrics/ND[0.5],eval_metrics/mean_weighted_sum_quantile_loss,domain,num_variates
2,bizitobs_l2c/H/long,LiteSpecFormer,242.153224,242.153224,11.451097,1.148851,1.327829,0.984141,29.264530,15.561273,0.950527,0.699466,0.643335,Web/CloudOps,7
1,bizitobs_l2c/H/medium,LiteSpecFormer,271.122563,271.122563,12.014438,1.167538,1.178029,1.045693,23.651005,16.465800,0.997029,0.727492,0.637512,Web/CloudOps,7
0,bizitobs_l2c/H/short,LiteSpecFormer,221.341394,221.341394,9.826005,0.943449,0.827046,0.951956,9.412636,14.877547,0.801938,0.529647,0.428380,Web/CloudOps,7
3,m4_weekly/W/short,LiteSpecFormer,322019.887722,322019.887722,287.601189,2.430750,0.070254,0.068906,15.811730,567.467962,0.103384,0.052396,0.041304,Econ/Fin,1
